# Exercise: Implementing a Simple Model Executor

Welcome! In this exercise, you'll build a simplified version of a model executor, a key component in an LLM inference engine like vLLM. The executor is the bridge between the high-level scheduler and the low-level model code (e.g., CUDA kernels). It takes a batch of sequences, runs them through the model, and manages the underlying resources like the KV cache.

By completing this exercise, you will:
*   Implement a method to simulate a model's forward pass to generate new tokens.
*   Implement a method to manage a simplified key-value (KV) cache for active sequences.
*   Orchestrate the end-to-end model execution for a batch of sequences, tying all the components together.

Let's get started!

In [ ]:
import dataclasses
from typing import List, Dict, Tuple, Set

# --- Helper Code (Do not modify) ---

# A special token ID that signifies the end of a sequence.
STOP_TOKEN_ID = 0
# The maximum number of tokens a sequence can have.
# Our simple model will generate the STOP_TOKEN_ID when this is reached.
MAX_TOKENS = 10

@dataclasses.dataclass
class SequenceData:
    """Represents the state of a single sequence (request)."""
    seq_id: int
    token_ids: List[int]
    completed: bool = False

    def __repr__(self):
        return f"Seq(id={self.seq_id}, tokens={self.token_ids}, completed={self.completed})"

# A batch is a list of sequences that the executor will process in one step.
Batch = List[SequenceData]

# --- End Helper Code ---

class SimpleModelExecutor:
    """
    A mock executor that simulates running inference for a batch of sequences.

    This class simulates the core logic of an LLM executor:
    1. It takes a batch of sequences.
    2. It runs a "model" to generate the next token for each sequence.
    3. It updates a simplified representation of the KV cache.
    """
    def __init__(self):
        """Initializes the executor and its KV cache."""
        # The KV cache is represented as a set of sequence IDs that are
        # currently "loaded" in the cache.
        self.kv_cache: Set[int] = set()

    # You will implement the following three methods in the exercises below.
    def _simulate_model_forward(self, last_token_ids: List[int]) -> List[int]:
        raise NotImplementedError("You must implement this method in Exercise 1.")

    def _update_kv_cache(self, seqs_to_add: List[int], seqs_to_remove: List[int]) -> None:
        raise NotImplementedError("You must implement this method in Exercise 2.")

    def execute_model(self, batch: Batch) -> None:
        raise NotImplementedError("You must implement this method in Exercise 3.")

### Exercise 1: Simulate the Model's Forward Pass

Your first task is to implement the `_simulate_model_forward` method. This function is the heart of the executor, representing the actual model inference. For this exercise, our "model" is very simple:

*   It takes a list of the *last token ID* from each sequence in the batch.
*   For each token, it generates the next token by simply adding 1 (`next_token = last_token + 1`).
*   If generating the next token would cause the sequence to exceed `MAX_TOKENS`, it should instead generate the `STOP_TOKEN_ID` to signal that the sequence is complete.

**Hint:** A list comprehension is a concise way to solve this, but a `for` loop works just as well! Remember to check the condition for generating a `STOP_TOKEN_ID`.

In [ ]:
def _simulate_model_forward(self, last_token_ids: List[int]) -> List[int]:
    """
    Simulates a forward pass of the model.

    Args:
        last_token_ids: A list of the last token ID for each sequence in the batch.

    Returns:
        A list of newly generated token IDs, one for each input sequence.
    """
    ### START CODE HERE ###
    # your code here
    pass
    ### END CODE HERE ###

# Monkey-patch the method into our class for testing
SimpleModelExecutor._simulate_model_forward = _simulate_model_forward

In [ ]:
# === Test Cell for Exercise 1 ===
executor = SimpleModelExecutor()

# Test case 1: Basic generation
last_tokens_1 = [5, 2, 8]
expected_next_tokens_1 = [6, 3, 9]
result_1 = executor._simulate_model_forward(last_tokens_1)
assert result_1 == expected_next_tokens_1, f"Test Case 1 Failed: Got {result_1}, expected {expected_next_tokens_1}"

# Test case 2: Generation with a stop condition
# A token ID of 9 means the sequence has 9 tokens. MAX_TOKENS is 10.
# The next token would make the length 10, so it should be a STOP_TOKEN_ID (0).
last_tokens_2 = [3, 9, 6]
expected_next_tokens_2 = [4, 0, 7] # Note the 0 for the second sequence
result_2 = executor._simulate_model_forward(last_tokens_2)
assert result_2 == expected_next_tokens_2, f"Test Case 2 Failed: Got {result_2}, expected {expected_next_tokens_2}"

# Test case 3: Empty input
last_tokens_3 = []
expected_next_tokens_3 = []
result_3 = executor._simulate_model_forward(last_tokens_3)
assert result_3 == expected_next_tokens_3, f"Test Case 3 Failed: Got {result_3}, expected {expected_next_tokens_3}"


print("✅ All tests for Exercise 1 passed!")

### Exercise 2: Update the KV Cache

Great job! Now, let's implement the logic for managing the KV cache. In a real system, this is a complex operation involving memory allocation on the GPU. Here, we'll simplify it greatly.

Our executor has an attribute `self.kv_cache`, which is a `set` containing the IDs of all sequences that are currently "active" in the cache.

Your task is to implement the `_update_kv_cache` method. It should:
*   Add all sequence IDs from `seqs_to_add` to `self.kv_cache`.
*   Remove all sequence IDs from `seqs_to_remove` from `self.kv_cache`.

**Hint:** Python's `set` objects have very efficient methods for adding and removing multiple items at once. Check out the `.update()` and `.difference_update()` methods!

In [ ]:
def _update_kv_cache(self, seqs_to_add: List[int], seqs_to_remove: List[int]) -> None:
    """
    Updates the set of sequence IDs present in the KV cache.

    Args:
        seqs_to_add: A list of sequence IDs to add to the cache.
        seqs_to_remove: A list of sequence IDs to remove from the cache.
    """
    ### START CODE HERE ###
    # your code here
    pass
    ### END CODE HERE ###

# Monkey-patch the method into our class for testing
SimpleModelExecutor._update_kv_cache = _update_kv_cache

In [ ]:
# === Test Cell for Exercise 2 ===
executor = SimpleModelExecutor()
assert executor.kv_cache == set(), "Initial KV cache should be empty"

# Test case 1: Adding new sequences
executor._update_kv_cache(seqs_to_add=[101, 102], seqs_to_remove=[])
expected_cache_1 = {101, 102}
assert executor.kv_cache == expected_cache_1, f"Test Case 1 Failed: Got {executor.kv_cache}, expected {expected_cache_1}"

# Test case 2: Removing a sequence
executor._update_kv_cache(seqs_to_add=[], seqs_to_remove=[101])
expected_cache_2 = {102}
assert executor.kv_cache == expected_cache_2, f"Test Case 2 Failed: Got {executor.kv_cache}, expected {expected_cache_2}"

# Test case 3: Adding and removing in the same step
executor._update_kv_cache(seqs_to_add=[103, 104], seqs_to_remove=[102])
expected_cache_3 = {103, 104}
assert executor.kv_cache == expected_cache_3, f"Test Case 3 Failed: Got {executor.kv_cache}, expected {expected_cache_3}"

# Test case 4: Removing a non-existent sequence (should not error)
executor._update_kv_cache(seqs_to_add=[], seqs_to_remove=[999])
assert executor.kv_cache == expected_cache_3, f"Test Case 4 Failed: Got {executor.kv_cache}, expected {expected_cache_3}"

print("✅ All tests for Exercise 2 passed!")

### Exercise 3: Orchestrate the Execution

Fantastic! You've built the two key helper methods. Now it's time to bring them together in the main `execute_model` method. This method orchestrates the entire process for a given batch.

Here are the steps you need to implement:

1.  **Filter Running Sequences**: Iterate through the input `batch`. Create a new list containing only the `SequenceData` objects that are **not** yet `completed`.
2.  **Handle Empty Batch**: If there are no running sequences, there's nothing to do, so you can `return` early.
3.  **Prepare for Model**: From your list of running sequences, extract the last token from each one to create a `last_token_ids` list.
4.  **Run Model**: Call `self._simulate_model_forward()` with the `last_token_ids` to get the `next_token_ids`.
5.  **Update Sequences**: Iterate through the `running_sequences` and `next_token_ids` together (the `zip` function is perfect for this!). For each sequence:
    *   Append the new token to its `token_ids` list.
    *   If the new token is the `STOP_TOKEN_ID`, set the sequence's `completed` flag to `True`.
6.  **Update Cache**: Determine which sequences to add to and remove from the cache.
    *   All running sequences were processed, so their IDs should be added to the cache.
    *   Sequences that just completed should be removed from the cache.
    *   Call `self._update_kv_cache()` with the appropriate lists of sequence IDs.

In [ ]:
def execute_model(self, batch: Batch) -> None:
    """
    Executes a single model step on a batch of sequences.

    Args:
        batch: A list of SequenceData objects to process.
    """
    ### START CODE HERE ###
    # your code here
    pass
    ### END CODE HERE ###

# Monkey-patch the final method into our class
SimpleModelExecutor.execute_model = execute_model

In [ ]:
# === Integration Test for Exercise 3 ===
# This test wires everything together.

# Step 1: Initialize sequences and executor
seq1 = SequenceData(seq_id=1, token_ids=[1, 2, 3])
seq2 = SequenceData(seq_id=2, token_ids=[1, 2, 3, 4, 5, 6, 7, 8, 9]) # This one will complete
seq3 = SequenceData(seq_id=3, token_ids=[1], completed=True) # Already completed
batch = [seq1, seq2, seq3]

executor = SimpleModelExecutor()

# Step 2: Run the first execution step
print("--- Running execution step 1 ---")
executor.execute_model(batch)

# Check sequence 1
expected_tokens_s1 = [1, 2, 3, 4]
assert seq1.token_ids == expected_tokens_s1, f"Seq1 tokens incorrect: Got {seq1.token_ids}, expected {expected_tokens_s1}"
assert not seq1.completed, "Seq1 should not be completed yet"

# Check sequence 2
expected_tokens_s2 = [1, 2, 3, 4, 5, 6, 7, 8, 9, 0]
assert seq2.token_ids == expected_tokens_s2, f"Seq2 tokens incorrect: Got {seq2.token_ids}, expected {expected_tokens_s2}"
assert seq2.completed, "Seq2 should now be completed"

# Check sequence 3 (should be unchanged)
expected_tokens_s3 = [1]
assert seq3.token_ids == expected_tokens_s3, f"Seq3 should not have changed: Got {seq3.token_ids}, expected {expected_tokens_s3}"
assert seq3.completed, "Seq3 should remain completed"

# Check KV cache state
# Seq1 was processed and is running. Seq2 was processed and finished.
# Both were "in the cache" for this step, but Seq2 was removed after.
expected_cache = {1}
assert executor.kv_cache == expected_cache, f"KV cache is incorrect after step 1: Got {executor.kv_cache}, expected {expected_cache}"
print("Step 1 checks passed!")


# Step 3: Run a second execution step
print("\n--- Running execution step 2 ---")
executor.execute_model(batch)

# Now, only seq1 should be processed.
expected_tokens_s1_step2 = [1, 2, 3, 4, 5]
assert seq1.token_ids == expected_tokens_s1_step2, f"Seq1 tokens incorrect after step 2: Got {seq1.token_ids}, expected {expected_tokens_s1_step2}"

# Seq2 and Seq3 should remain unchanged from the end of step 1
assert seq2.token_ids == expected_tokens_s2, f"Seq2 should not have changed in step 2"
assert seq3.token_ids == expected_tokens_s3, f"Seq3 should not have changed in step 2"

# Check KV cache state again
# Seq1 was processed again, so it remains.
expected_cache_step2 = {1}
assert executor.kv_cache == expected_cache_step2, f"KV cache is incorrect after step 2: Got {executor.kv_cache}, expected {expected_cache_step2}"
print("Step 2 checks passed!")

print("\n✅ Congratulations! All exercises and tests passed. You've successfully implemented a simple model executor!")

### Optional Challenge

Well done on completing the core exercises! If you're looking for an extra challenge, consider how you might modify the `SimpleModelExecutor` to handle more advanced scenarios:

*   **Preemption:** What if a high-priority request arrives? The scheduler might decide to "preempt" (pause) a lower-priority sequence. How would you change `execute_model` and `_update_kv_cache` to handle a list of sequences that should be removed from the cache *even if they haven't finished generating*?
*   **Varying Output Lengths**: Our model generates exactly one token per sequence per step. Real models can have varying computation times. How could you simulate this? Maybe `_simulate_model_forward` could randomly decide not to return a token for some sequences, simulating a longer processing time. How would the rest of the logic need to adapt?

These are complex topics at the heart of why systems like vLLM are so powerful. Thinking about them will give you an even deeper appreciation for the challenges of building high-performance LLM inference engines.